# V7 Trainable Quantum Rerun (Clean Colab Notebook)

This notebook is the clean Colab path for the April 2026 V7 confirmatory rerun.

Current purpose:
- produce a fresh artifact-backed V7 run with a top-level `experiments/*.json` row
- keep Drive-backed checkpoints
- allow resume after disconnection
- continue beyond an early target hit by rerunning with a higher `--target`
- copy the JSON row and experiment directory back to Drive


## 1. Runtime Check

Use an L4 if available. A100 is also acceptable.


In [1]:
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


Mon Apr 27 19:30:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   38C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Drive


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 3. Clone or Refresh Repo

If the folder already exists, this cell refreshes it.


In [3]:
import os

repo_path = '/content/Quanvolutional-Neural-Network'
if not os.path.exists(repo_path):
    !git clone https://github.com/necatiincekara/Quanvolutional-Neural-Network.git /content/Quanvolutional-Neural-Network
else:
    %cd /content/Quanvolutional-Neural-Network
    !git pull

%cd /content/Quanvolutional-Neural-Network
!git branch
!git log --oneline -n 3


Cloning into '/content/Quanvolutional-Neural-Network'...
remote: Enumerating objects: 595, done.
remote: Counting objects: 100% (595/595), done.
remote: Compressing objects: 100% (355/355), done.
remote: Total 595 (delta 294), reused 491 (delta 190), pack-reused 0 (from 0)
Receiving objects: 100% (595/595), 697.96 KiB | 16.23 MiB/s, done.
Resolving deltas: 100% (294/294), done.
/content/Quanvolutional-Neural-Network
* master
cede472 (HEAD -> master, origin/master, origin/HEAD) update colab_v7_rerun_clean
13c0cba Harden V7 artifacts and Codex worklfow
c6f77b8 Add modern classical baselines and update benchmark results


## 4. Install Dependencies


In [4]:
%cd /content/Quanvolutional-Neural-Network
!pip install -r requirements.txt


/content/Quanvolutional-Neural-Network
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 140.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.3/913.3 kB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 162.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 MB 37.6 MB/s eta 0:00:00:00:0100:01


## 5. Stage Dataset To Local Colab Disk

Drive remains the source of truth; local disk is used for speed.


In [5]:
!mkdir -p /content/local_data/train /content/local_data/test
!rsync -a /content/drive/MyDrive/set/train/ /content/local_data/train/
!rsync -a /content/drive/MyDrive/set/test/ /content/local_data/test/
!find /content/local_data/train -type f | wc -l
!find /content/local_data/test -type f | wc -l


3428
466


## 6. Sanity Check CLI

The updated entrypoint should support `--resume`, `--drive-backup-path`, `--train-path`, `--test-path`, and `--result-json`.


In [6]:
%cd /content/Quanvolutional-Neural-Network
!python train_v7.py --help


/content/Quanvolutional-Neural-Network
usage: train_v7.py [-h]
                   [--circuit {data_reuploading,strongly_entangling,hardware_efficient}]
                   [--epochs EPOCHS] [--target TARGET] [--resume]
                   [--drive-backup-path DRIVE_BACKUP_PATH]
                   [--train-path TRAIN_PATH] [--test-path TEST_PATH]
                   [--result-json RESULT_JSON]

V7 Enhanced Quantum Training

options:
  -h, --help            show this help message and exit
  --circuit {data_reuploading,strongly_entangling,hardware_efficient}
                        Quantum circuit type
  --epochs EPOCHS       Number of training epochs (default: 3)
  --target TARGET       Target validation accuracy (default: 60.0)
  --resume              Resume from latest local or Drive-backed checkpoint
  --drive-backup-path DRIVE_BACKUP_PATH
                        Optional Drive path for persistent checkpoint backup
  --train-path TRAIN_PATH
                        Override training datas

## 7. Clean Fresh Confirmatory Run

Use this first. It intentionally clears local V7 checkpoints so the run starts from scratch.


In [7]:
%cd /content/Quanvolutional-Neural-Network
!mkdir -p /content/drive/MyDrive/quanv_results/v7_clean_20260427/experiments
!rm -f models/checkpoint_latest_v7.pth models/best_v7_model.pth
!python train_v7.py \
  --circuit data_reuploading \
  --epochs 10 \
  --target 90 \
  --drive-backup-path /content/drive/MyDrive/quanv_results/v7_clean_20260427 \
  --train-path /content/local_data/train \
  --test-path /content/local_data/test \
  --result-json experiments/v7_trainable_quantum_clean_20260427_l4.json


/content/Quanvolutional-Neural-Network
V7 ENHANCED QUANTUM TRAINING
Platform:        Colab
Compute Device:  cuda
Quantum Device:  lightning.gpu
Circuit Type:    data_reuploading
Epochs:          10
Target Accuracy: 90.0%
Resume:          False
Train Path:      /content/local_data/train
Test Path:       /content/local_data/test
Drive Backup:    /content/drive/MyDrive/quanv_results/v7_clean_20260427
Result JSON:     experiments/v7_trainable_quantum_clean_20260427_l4.json
GPU:             NVIDIA L4
GPU Memory:      23.7 GB
Loading images from: /content/local_data/train
Loaded 3428 images.
Loading images from: /content/local_data/test
Loaded 466 images.
Quantum params: 25
Classical params: 87773

V7 ENHANCED TRAINING
Target accuracy: 90.0%
V4 baseline: 8.75%
Experiment: experiments/v7_data_reuploading_20260427_203430
Device: cuda
Epochs: 1 -> 10


--- Epoch 1/10 ---
Epoch 1:   0% 0/25 [00:00<?, ?it/s]  [DEBUG] gradient_scale = 0.1000
  [DEBUG] quantum grad mean=7.94e-03, max=1.51e-02
  [DE

## 8. Continue Past Early Target Hit

If training stops early because the target was reached, rerun with a higher target and `--resume`.
Example below continues the same clean run toward the full 10-epoch budget.


In [ ]:
%cd /content/Quanvolutional-Neural-Network
!python train_v7.py \
  --circuit data_reuploading \
  --epochs 10 \
  --target 90 \
  --resume \
  --drive-backup-path /content/drive/MyDrive/quanv_results/v7_clean_20260427 \
  --train-path /content/local_data/train \
  --test-path /content/local_data/test \
  --result-json experiments/v7_trainable_quantum_clean_20260427_l4.json


## 9. Inspect Saved Artifacts


In [ ]:
!RUN_DIR=/content/drive/MyDrive/quanv_results/v7_clean_20260427; \
  cp experiments/v7_trainable_quantum_clean_20260427_l4.json "$RUN_DIR/"; \
  rsync -a experiments/v7_data_reuploading_*/ "$RUN_DIR/experiments/"; \
  ls -lah models; \
  ls -lah "$RUN_DIR"; \
  ls -lah "$RUN_DIR/experiments"; \
  python -m json.tool "$RUN_DIR/v7_trainable_quantum_clean_20260427_l4.json" | head -80


## 10. Recovery After Runtime Disconnect

Use this only if the training finished but the runtime disconnected before artifact-copy cell 9 ran. This does not retrain when the Drive-backed latest checkpoint is at epoch 10; it restores the checkpoint, runs final test evaluation, emits the JSON row, and copies artifacts back to Drive.


In [ ]:
%cd /content/Quanvolutional-Neural-Network
!mkdir -p /content/local_data/train /content/local_data/test
!rsync -a /content/drive/MyDrive/set/train/ /content/local_data/train/
!rsync -a /content/drive/MyDrive/set/test/ /content/local_data/test/
!mkdir -p /content/drive/MyDrive/quanv_results/v7_clean_20260427/experiments
!python train_v7.py \
  --circuit data_reuploading \
  --epochs 10 \
  --target 90 \
  --resume \
  --drive-backup-path /content/drive/MyDrive/quanv_results/v7_clean_20260427 \
  --train-path /content/local_data/train \
  --test-path /content/local_data/test \
  --result-json experiments/v7_trainable_quantum_clean_20260427_l4.json
!RUN_DIR=/content/drive/MyDrive/quanv_results/v7_clean_20260427; \
  cp experiments/v7_trainable_quantum_clean_20260427_l4.json "$RUN_DIR/"; \
  cp models/best_v7_model.pth "$RUN_DIR/" || true; \
  cp models/checkpoint_latest_v7.pth "$RUN_DIR/" || true; \
  rsync -a experiments/v7_data_reuploading_*/ "$RUN_DIR/experiments/" || true; \
  ls -lah "$RUN_DIR"; \
  python -m json.tool "$RUN_DIR/v7_trainable_quantum_clean_20260427_l4.json" | head -120
